In [ ]:
import tkinter as tk
from tkinter import messagebox

# ==========================================================
# SmartPack - Bin Packing Optimizer
# DAA Mini Project
# Real-life use: warehouse/package loading
# ==========================================================

def first_fit(items, capacity):
    bins = []
    contents = []

    for item in items:
        placed = False

        for i, space in enumerate(bins):
            if space >= item:
                bins[i] -= item
                contents[i].append(item)
                placed = True
                break

        if not placed:
            bins.append(capacity - item)
            contents.append([item])

    return contents


def first_fit_decreasing(items, capacity):
    return first_fit(sorted(items, reverse=True), capacity)


def best_fit_decreasing(items, capacity):
    items = sorted(items, reverse=True)
    spaces = []
    contents = []

    for item in items:
        best = -1
        smallest_remaining = float("inf")

        for i, space in enumerate(spaces):
            remaining = space - item

            if remaining >= 0 and remaining < smallest_remaining:
                smallest_remaining = remaining
                best = i

        if best != -1:
            spaces[best] -= item
            contents[best].append(item)
        else:
            spaces.append(capacity - item)
            contents.append([item])

    return contents


def run_algorithm():
    try:
        items = [float(x.strip()) for x in item_entry.get().split(",")]
        capacity = float(capacity_entry.get())

        if not items or capacity <= 0:
            raise ValueError

        if any(x <= 0 or x > capacity for x in items):
            messagebox.showerror(
                "Invalid Input",
                "Each package size must be greater than 0 and not exceed bin capacity."
            )
            return

        total = sum(items)
        lower_bound = int((total + capacity - 0.000001) // capacity)

        ff = first_fit(items, capacity)
        ffd = first_fit_decreasing(items, capacity)
        bfd = best_fit_decreasing(items, capacity)

        show_results(items, capacity, total, lower_bound, ff, ffd, bfd)

    except ValueError:
        messagebox.showerror(
            "Invalid Input",
            "Enter package sizes like:\n0.5, 0.7, 0.3, 0.9"
        )


def show_results(items, capacity, total, lower, ff, ffd, bfd):
    result.delete("1.0", tk.END)

    result.insert(tk.END, "SMARTPACK OPTIMIZATION RESULT\n")
    result.insert(tk.END, "=" * 65 + "\n\n")

    result.insert(tk.END, f"Total package size : {total:.2f}\n")
    result.insert(tk.END, f"Bin capacity       : {capacity:.2f}\n")
    result.insert(tk.END, f"Theoretical lower bound : {lower} bins\n\n")

    algorithms = [
        ("FIRST FIT (FF)", ff),
        ("FIRST FIT DECREASING (FFD)", ffd),
        ("BEST FIT DECREASING (BFD)", bfd)
    ]

    for name, bins in algorithms:
        result.insert(tk.END, f"{name}\n")
        result.insert(tk.END, "-" * 65 + "\n")

        for i, package_list in enumerate(bins, 1):
            used = sum(package_list)
            percentage = min(100, (used / capacity) * 100)
            bar = "█" * int(percentage / 5)
            result.insert(
                tk.END,
                f"Bin {i:02d}  {package_list}  "
                f"Used: {used:.2f}/{capacity:.2f}  [{bar}]\n"
            )

        result.insert(tk.END, f"Total bins: {len(bins)}\n\n")

    best_count = min(len(ff), len(ffd), len(bfd))

    if len(ffd) == best_count:
        best_name = "First Fit Decreasing (FFD)"
    elif len(bfd) == best_count:
        best_name = "Best Fit Decreasing (BFD)"
    else:
        best_name = "First Fit (FF)"

    result.insert(tk.END, "RECOMMENDATION\n")
    result.insert(tk.END, "-" * 65 + "\n")
    result.insert(
        tk.END,
        f"{best_name} uses the fewest bins in this test: {best_count}\n"
    )

    status.config(
        text=f"Status: Optimization complete | Best: {best_name}"
    )


def load_sample():
    item_entry.delete(0, tk.END)
    item_entry.insert(
        0,
        "0.5,0.7,0.3,0.9,0.2,0.6,0.8,0.4,0.1,0.5"
    )
    capacity_entry.delete(0, tk.END)
    capacity_entry.insert(0, "1.0")
    status.config(text="Status: Sample data loaded")


def clear():
    item_entry.delete(0, tk.END)
    result.delete("1.0", tk.END)
    status.config(text="Status: Ready")


# ===================== UI =====================

root = tk.Tk()
root.title("SmartPack - Bin Packing Optimizer")
root.geometry("950x720")
root.configure(bg="#EEF4FF")

# Header
header = tk.Frame(root, bg="#173F5F", height=90)
header.pack(fill="x")

tk.Label(
    header,
    text="SMARTPACK",
    font=("Segoe UI", 24, "bold"),
    bg="#173F5F",
    fg="white"
).pack(pady=(12, 0))

tk.Label(
    header,
    text="Bin Packing Optimization • Design & Analysis of Algorithms",
    font=("Segoe UI", 10),
    bg="#173F5F",
    fg="#D7F3FF"
).pack()

# Real-world description
tk.Label(
    root,
    text="📦 Real-Life Application: Optimize warehouse/package loading "
         "and reduce the number of containers used.",
    font=("Segoe UI", 11, "bold"),
    bg="#EEF4FF",
    fg="#173F5F"
).pack(pady=12)

# Input card
input_card = tk.Frame(root, bg="white", bd=1, relief="solid")
input_card.pack(fill="x", padx=30, pady=5)

tk.Label(
    input_card,
    text="Package Sizes",
    font=("Segoe UI", 10, "bold"),
    bg="white",
    fg="#173F5F"
).grid(row=0, column=0, padx=15, pady=(15, 5), sticky="w")

item_entry = tk.Entry(
    input_card,
    font=("Consolas", 11),
    width=80,
    bg="#F8FBFF"
)
item_entry.grid(row=1, column=0, columnspan=3, padx=15, pady=5)

tk.Label(
    input_card,
    text="Container Capacity",
    font=("Segoe UI", 10, "bold"),
    bg="white",
    fg="#173F5F"
).grid(row=2, column=0, padx=15, pady=(10, 5), sticky="w")

capacity_entry = tk.Entry(
    input_card,
    font=("Consolas", 11),
    width=15,
    bg="#F8FBFF"
)
capacity_entry.grid(row=3, column=0, padx=15, pady=(0, 15), sticky="w")

# Buttons
button_frame = tk.Frame(root, bg="#EEF4FF")
button_frame.pack(pady=12)

tk.Button(
    button_frame,
    text="📊 Optimize",
    command=run_algorithm,
    bg="#16A085",
    fg="white",
    font=("Segoe UI", 11, "bold"),
    width=17,
    relief="flat",
    cursor="hand2"
).grid(row=0, column=0, padx=6)

tk.Button(
    button_frame,
    text="📦 Sample Data",
    command=load_sample,
    bg="#F39C12",
    fg="white",
    font=("Segoe UI", 11, "bold"),
    width=17,
    relief="flat",
    cursor="hand2"
).grid(row=0, column=1, padx=6)

tk.Button(
    button_frame,
    text="Clear",
    command=clear,
    bg="#E74C3C",
    fg="white",
    font=("Segoe UI", 11, "bold"),
    width=17,
    relief="flat",
    cursor="hand2"
).grid(row=0, column=2, padx=6)

# Result
result_card = tk.LabelFrame(
    root,
    text="Optimization Analysis",
    font=("Segoe UI", 11, "bold"),
    bg="white",
    fg="#173F5F",
    padx=10,
    pady=10
)
result_card.pack(fill="both", expand=True, padx=30, pady=5)

result = tk.Text(
    result_card,
    font=("Consolas", 10),
    bg="#FBFDFF",
    fg="#263238",
    wrap="none",
    relief="flat"
)
result.pack(fill="both", expand=True)

# Status
status = tk.Label(
    root,
    text="Status: Ready",
    bg="#173F5F",
    fg="white",
    anchor="w",
    padx=15,
    pady=7,
    font=("Segoe UI", 9)
)
status.pack(fill="x", side="bottom")

load_sample()
root.mainloop()